# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Importing Libraries</p>

In [ ]:
!pip install -q hillclimbers

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np 
import pandas as pd
import seaborn as sns 
from scipy import stats
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from category_encoders import TargetEncoder
from hillclimbers import climb_hill, partial
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibrationDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import mutual_info_classif

oof = {}
test_pred = {}
NUM_FOLD = 5
target = 'loan_status'

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Loading Data</p>

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s4e10/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/playground-series-s4e10/test.csv', index_col='id')
original = pd.read_csv('/kaggle/input/loan-approval-prediction/credit_risk_dataset.csv')
sample_submission = pd.read_csv('/kaggle/input/playground-series-s4e10/sample_submission.csv')
train.head()

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Data Exploration</p>

In [ ]:
train.info() 

In [ ]:
test.info()

In [ ]:
original.info() 

In [ ]:
X_ = train.copy()
y_ = X_.pop("loan_status")

for col in X_.select_dtypes("object"):
    X_[col], _ = X_[col].factorize()

discrete_features = X_.dtypes == int

mi_scores = mutual_info_classif(X_, y_, discrete_features=discrete_features)
mi_scores = pd.Series(mi_scores, name="MI Scores", index=X_.columns)
mi_scores = mi_scores.sort_values(ascending=True)

width = np.arange(len(mi_scores))
ticks = list(mi_scores.index)
plt.barh(width, mi_scores)
plt.yticks(width, ticks)
plt.title("Mutual Information Scores")
plt.figure(dpi=100, figsize=(8, 5))

In [ ]:
f, axes = plt.subplots(5, 2, figsize=(14, 24)) 
col = test.columns.tolist()
col.remove('loan_intent')
j = 0
for i in range(0, 10, 2):
    sns.histplot(data=train, x=col[i], element="step", hue="loan_status", ax=axes[j, 0]) 
    sns.histplot(data=train, x=col[i+1], element="step", hue="loan_status", ax=axes[j, 1])
    j = j + 1

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Cross Validation</p>

In [ ]:
def cross_validation(model, label):
    
    train_copy, test_copy, original_copy = train.copy(), test.copy(), original.copy()
    
    if label in ['cb', 'et', 'rf', 'knn', 'mlp']:
        cat_cols = test_copy.columns.tolist()
        for df in [train_copy, test_copy, original_copy]:
            for col in cat_cols:  
                df[col] = df[col].astype('str').astype('category')
        
    elif label in ['xgb', 'lgbm', 'dart', 'goss']: 
        cat_cols = list(test_copy.select_dtypes(include=['object']).columns)
        for df in [train_copy, test_copy, original_copy]:
            for col in cat_cols:  
                df[col] = df[col].astype('str').astype('category')
                
    X = train_copy.drop([target], axis=1)
    y = train_copy[target]
    X_original = original_copy.drop([target], axis=1)
    y_original = original_copy[target]
        

    val_scores = []
    test_preds_model = []
    oof_model = np.zeros(len(train),)
    
    skf = StratifiedKFold(n_splits=NUM_FOLD, shuffle=True, random_state=1)

    for Fold, (train_index, val_index) in enumerate(skf.split(X, y)):
    
        X_train, X_val = X.iloc[train_index], X.iloc[val_index]
        y_train, y_val = y[train_index], y[val_index]
        
        X_train = pd.concat([X_train, X_original], axis=0)
        y_train = pd.concat([y_train, y_original]) 
    
        model.fit(X_train, y_train)
    
        y_pred = model.predict_proba(X_val)[:, 1]
    
        roc_auc_score_ = roc_auc_score(y_val, y_pred)
    
        print(f'Fold {Fold}: roc_auc_score= {roc_auc_score_:.5f}')
    
        val_scores.append(roc_auc_score_)
        
        oof_model[val_index] = y_pred
        
        test_preds_model.append(model.predict_proba(test_copy)[:, 1])
    
    oof[label] = oof_model
    
    test_preds_model = sum(test_preds_model)/len(test_preds_model)
    test_pred[label] = test_preds_model 

    print(f'mean validation roc_auc_score = {np.mean(val_scores):.5f}')
    print(f'std validation roc_auc_score = {np.std(val_scores):.5f}')
    
    plt.figure(figsize=(10, 4))
    plt.suptitle(label, y=1.0, fontsize=20)
    ax = plt.subplot(1, 2, 1)
    plt.title('Calibration')
    CalibrationDisplay.from_predictions(y, oof_model, n_bins=10, strategy='quantile', ax=ax)
    plt.subplot(1, 2, 2)
    plt.title('Histogram')
    plt.hist(oof_model, bins=10)
    plt.show()

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Lightgbm</p>

In [ ]:
%%time

params_lgbm = {
    
    'verbose': -1,
    'random_state': 1,
    'objective': 'binary',
    'n_estimators': 4100,
    'learning_rate': 0.01,
    'colsample_bytree': 0.6,
    'max_depth': 8,
    'max_bin': 5000,
}

model_1 = LGBMClassifier(**params_lgbm)

cross_validation(model_1, 'lgbm')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Dart</p>

In [ ]:
%%time

params_dart = {
    
    'verbose': -1,
    'random_state': 1,
    'boosting': 'dart',
    'n_estimators': 600,
    'learning_rate': 0.1,
    'colsample_bytree': 0.6,
    'num_leaves': 85,
    'min_data_in_leaf': 30,
    'max_bin': 1995,
    'objective': 'binary',
}

model_2 = LGBMClassifier(**params_dart)

cross_validation(model_2, 'dart')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">XGBoost</p>

In [ ]:
%%time

params_xgb = {
    
    'enable_categorical': True,
    'random_state': 1,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'colsample_bytree': 0.6,
    'reg_lambda': 0.01,
    'max_depth': 4,
    'max_bin': 5000,
    'subsample': 0.95,
    'reg_alpha': 0.1,
 
}

model_3 = XGBClassifier(**params_xgb)

cross_validation(model_3, 'xgb')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">ExtraTrees</p>

In [ ]:
%%time

params_et = {
    
    'random_state': 1,
    'n_estimators': 470,
    'min_samples_leaf': 1,
    'max_depth': 20,
    'criterion': 'log_loss',
}

model_4 = make_pipeline(TargetEncoder(), ExtraTreesClassifier(**params_et))

cross_validation(model_4, 'et')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Random Forest</p>

In [ ]:
%%time

params_rf = {
    
    'random_state': 1,
    'n_estimators': 450,
    'min_samples_leaf': 5,
    'max_leaf_nodes': 960,
    'criterion': 'entropy',
}

model_5 = make_pipeline(TargetEncoder(), RandomForestClassifier(**params_rf))

cross_validation(model_5, 'rf')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">CatBoost</p>

In [ ]:
%%time

params_cb = {
    
    'verbose': False,
    'random_state': 1,
    'task_type': 'CPU',
    'cat_features' : test.columns.tolist(),
    'min_data_in_leaf': 5,
    'n_estimators': 1800,
    'random_strength': 0.79,
    'depth': 8,
    'bagging_temperature': 0.6,
    'l2_leaf_reg': 4,
    'rsm': 0.6,
}

model_6 = CatBoostClassifier(**params_cb)

cross_validation(model_6, 'cb')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">KNeighbors</p>

In [ ]:
%%time

model_7 = make_pipeline(TargetEncoder(), KNeighborsClassifier(n_neighbors=185, 
                                                              metric='manhattan',
                                                              weights='distance'))

cross_validation(model_7, 'knn')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">MLP</p>

In [ ]:
%%time

params_mlp = {
    
    'random_state': 1,
    'hidden_layer_sizes': (32, 3),
    
}

model_8 = make_pipeline(TargetEncoder(), StandardScaler(), MLPClassifier(**params_mlp))

cross_validation(model_8, 'mlp')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Goss</p>

In [ ]:
%%time

params_goss = {
    
    'verbose': -1,
    'random_state': 1,
    'data_sample_strategy': 'goss',
    'n_estimators': 4000,
    'learning_rate': 0.01,
    'colsample_bytree': 0.6,
    'max_depth': 17,
    'max_bin': 4000,
}

model_9 = LGBMClassifier(**params_goss)

cross_validation(model_9, 'goss')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">ECDF Plot</p>

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))
ax.set(xscale="log", yscale="log")
sns.ecdfplot(pd.DataFrame({'xgboost': oof['xgb'], 'mlp': oof['mlp'], 'goss': oof['goss'],
                           'lightgbm': oof['lgbm'], 'dart': oof['dart'],
                           'extra_trees': oof['et'], 'random_forest': oof['rf'], 
                           'catboost': oof['cb'],'kneighbors': oof['knn']}), ax=ax)

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">(Logistic Regression) Ensemble</p>

In [ ]:
ensemble_fold_scores = []
ensemble_test_preds = []

y_ensemble = train[target]
X_ensemble = pd.DataFrame(oof)    
x_test_ensemble = pd.DataFrame(test_pred)

skf = StratifiedKFold(n_splits=NUM_FOLD, shuffle=True, random_state=1)

for i, (train_index_ens, val_index_ens) in enumerate(skf.split(X_ensemble, y_ensemble)):
    
    X_train_ens, X_val_ens = X_ensemble.iloc[train_index_ens], X_ensemble.iloc[val_index_ens]
    y_train_ens, y_val_ens = y_ensemble[train_index_ens], y_ensemble[val_index_ens]
    
    lr = LogisticRegression().fit(X_train_ens, y_train_ens)
    
    ensemble_val_pred = lr.predict_proba(X_val_ens)[:, 1] 
    
    ensemble_roc_auc_score = roc_auc_score(y_val_ens, ensemble_val_pred)
    ensemble_fold_scores.append(ensemble_roc_auc_score)
    
    print('Fold', i, '==> roc_auc_score (LR ensemble) is ==>', ensemble_roc_auc_score)
    
    ensemble_test_preds.append(lr.predict_proba(x_test_ensemble)[:, 1])
    
ensemble_test_preds = sum(ensemble_test_preds)/len(ensemble_test_preds)

print(f'\nCV roc_auc_score = {np.mean(ensemble_fold_scores):.5f}')
print(f'\nstd roc_auc_score = {np.std(ensemble_fold_scores):.5f}')

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Hill climbing</p>

In [ ]:
hc_test, hc_oof = climb_hill(train=train, target=target, objective='maximize', 
                             eval_metric=partial(roc_auc_score),oof_pred_df= X_ensemble, 
                             test_pred_df= x_test_ensemble,plot_hill=True,plot_hist=False, 
                             precision=0.001,negative_weights=True,return_oof_preds=True)

# <p style="background-color:#a7d14d;font-family:newtimeroman;color:#e609d3;font-size:150%;text-align:center;border-radius:20px 60px;">Submission</p>

In [ ]:
sample_submission[target] = hc_test
sample_submission.head()

In [ ]:
sample_submission.to_csv('submission.csv', index=False) 